# Radar Financeiro — estudo do ciclo após a execução

Execute este notebook depois que a tabela inicial estiver gravada. Informe `dt_exea_estudo` e a quantidade de ciclos usada pelo lote. O estudo considera todos os MCIs da data, conta divergências e agrupa os participantes por dia de sincronização, dia de ciclo e período. A última célula permite isolar o exemplo pelas datas e pelo dia cadastrado.

A leitura usa o conteúdo disponível em `SBX_A2A13E2.ANL_RADAR_FNC_CLI`. A gravação do Radar substitui o lote anterior, então o estudo de uma data histórica depende de os dados dessa execução ainda estarem disponíveis.

In [ ]:
from src.utils.gerenciador_local_v2 import GerenciadorLocal

gerenciador_local = GerenciadorLocal(
    nome_sessao="radar-estudo-ciclo",
    exibir_configuracao=False,
    ativar_logs=True,
)
spark = gerenciador_local.criar_sessao_spark(
    spark_conf={"spark.sql.session.timeZone": "America/Sao_Paulo"},
)

## Selecionar a execução

Use a data no formato `AAAA-MM-DD`. O recálculo usa a data de sincronização interpretada pelo Spark e o dia de ciclo que ficaram gravados. Use `contar_ultimo_dia = True` para lotes da V5 com a regra atual. Para lotes da V4 ou da V5 anterior ao ajuste, use `False`.

In [ ]:
%%spark
from datetime import date

from pyspark.sql import functions as F

dt_exea_estudo = "2026-11-25"
quantidade_ciclos_estudo = 1
contar_ultimo_dia = True

data_estudo = date.fromisoformat(dt_exea_estudo)
if quantidade_ciclos_estudo < 1:
    raise ValueError("Informe pelo menos um ciclo para o estudo.")


def calcular_datas_ciclo(quantidade_ciclos, contar_ultimo_dia):
    def inicio_ciclo_no_mes(mes, dia):
        return F.date_add(mes, F.least(dia, F.dayofmonth(F.last_day(mes))) - 1)

    data_sincronizacao = F.to_date("TS_INCL_TRAN_REF")
    data_referencia_ciclo = F.date_add(data_sincronizacao, 1) if contar_ultimo_dia else data_sincronizacao
    mes_referencia_ciclo = F.trunc(data_referencia_ciclo, "month")
    dia_ciclo = F.coalesce(F.col("DD_INC_MM_CLC_BLC_REF").cast("int"), F.lit(1))
    inicio_no_mes = inicio_ciclo_no_mes(mes_referencia_ciclo, dia_ciclo)
    inicio_ciclo_aberto = F.when(
        data_referencia_ciclo >= inicio_no_mes, inicio_no_mes
    ).otherwise(
        inicio_ciclo_no_mes(F.add_months(mes_referencia_ciclo, -1), dia_ciclo)
    )
    mes_inicio_periodo = F.add_months(F.trunc(inicio_ciclo_aberto, "month"), -quantidade_ciclos)

    return {
        "DT_SINCRONIZACAO_USADA": data_sincronizacao,
        "DIA_CICLO_APLICADO": dia_ciclo,
        "DT_INICIO_CICLO_ABERTO": inicio_ciclo_aberto,
        "DT_INI_REF": inicio_ciclo_no_mes(mes_inicio_periodo, dia_ciclo),
        "DT_FIM_REF": F.date_sub(inicio_ciclo_aberto, 1),
    }


base_gravada = spark.table("SBX_A2A13E2.ANL_RADAR_FNC_CLI").where(
    F.col("DT_EXEA") == F.lit(data_estudo)
)
datas_ciclo = calcular_datas_ciclo(quantidade_ciclos_estudo, contar_ultimo_dia)
participa = F.col("FLAG_PARTICIPA_RADAR") == "true"
inicio_esperado = F.when(participa, datas_ciclo["DT_INI_REF"])
fim_esperado = F.when(participa, datas_ciclo["DT_FIM_REF"])

analise_ciclo = base_gravada.select(
    "DT_EXEA",
    "FLAG_PARTICIPA_RADAR",
    "DD_INC_MM_CLC_BLC_REF",
    datas_ciclo["DIA_CICLO_APLICADO"].alias("DIA_CICLO_APLICADO"),
    datas_ciclo["DT_SINCRONIZACAO_USADA"].alias("DT_SINCRONIZACAO_USADA"),
    datas_ciclo["DT_INICIO_CICLO_ABERTO"].alias("DT_INICIO_CICLO_ABERTO"),
    "DT_INI_REF",
    "DT_FIM_REF",
    inicio_esperado.alias("DT_INI_REF_RECALCULADA"),
    fim_esperado.alias("DT_FIM_REF_RECALCULADA"),
    (
        F.col("DT_INI_REF").eqNullSafe(inicio_esperado)
        & F.col("DT_FIM_REF").eqNullSafe(fim_esperado)
    ).alias("PERIODO_CONFERE"),
)

## Resumo e períodos por data de sincronização

O resumo conta as linhas da execução. A distribuição mostra os participantes e as linhas com divergência, ordenadas primeiro pelo resultado da conferência. Os filtros da última célula são opcionais e afetam apenas a distribuição.

In [ ]:
%%spark
resumo_ciclo = analise_ciclo.agg(
    F.count("*").alias("REGISTROS"),
    F.count(F.when(F.col("FLAG_PARTICIPA_RADAR") == "true", 1)).alias("PARTICIPANTES"),
    F.count(F.when(~F.col("PERIODO_CONFERE"), 1)).alias("PERIODOS_DIVERGENTES"),
).first()

print(
    f"DT_EXEA = {data_estudo} | CICLOS = {quantidade_ciclos_estudo} | "
    f"ÚLTIMO DIA CONTA = {contar_ultimo_dia} | "
    f"FUSO SPARK = {spark.conf.get('spark.sql.session.timeZone')}"
)
if resumo_ciclo["REGISTROS"] == 0:
    print(f"Sem linhas gravadas para DT_EXEA = {data_estudo}.")
else:
    for nome, total in resumo_ciclo.asDict().items():
        print(f"{nome} = {total}")

In [ ]:
%%spark
dt_sincronizacao_filtro = None  # Exemplo: "2026-10-28".
dia_cadastrado_filtro = None  # Exemplo: 28.
dt_ini_gravada_filtro = None  # Exemplo: "2026-08-28".
dt_fim_gravada_filtro = None  # Exemplo: "2026-09-27".

if resumo_ciclo["REGISTROS"] > 0:
    recorte = analise_ciclo.where(
        (F.col("FLAG_PARTICIPA_RADAR") == "true") | ~F.col("PERIODO_CONFERE")
    )

    for campo, valor in (
        ("DT_SINCRONIZACAO_USADA", dt_sincronizacao_filtro),
        ("DD_INC_MM_CLC_BLC_REF", dia_cadastrado_filtro),
        ("DT_INI_REF", dt_ini_gravada_filtro),
        ("DT_FIM_REF", dt_fim_gravada_filtro),
    ):
        if valor is not None:
            if isinstance(valor, str):
                valor = date.fromisoformat(valor)
            recorte = recorte.where(F.col(campo) == F.lit(valor))

    (
        recorte.groupBy(
            "FLAG_PARTICIPA_RADAR",
            "DT_SINCRONIZACAO_USADA",
            "DD_INC_MM_CLC_BLC_REF",
            "DIA_CICLO_APLICADO",
            "DT_INICIO_CICLO_ABERTO",
            "DT_INI_REF",
            "DT_FIM_REF",
            "DT_INI_REF_RECALCULADA",
            "DT_FIM_REF_RECALCULADA",
            "PERIODO_CONFERE",
        )
        .count()
        .withColumnRenamed("count", "REGISTROS")
        .orderBy("PERIODO_CONFERE", F.desc("REGISTROS"), "DT_SINCRONIZACAO_USADA")
        .show(50, truncate=False)
    )